## IMDB Sentiment Analysis
*IMDB Sentiment Analysis is a common natural language processing (NLP) task where the goal is to classify movie reviews as either positive or negative based on the text content. The dataset consists of movie reviews from the IMDB website, typically labeled as either "positive" or "negative."*

#### Database Source
*The dataset is sourced directly from the Keras API, a popular deep learning library built on TensorFlow. Keras provides access to various preprocessed datasets commonly used for machine learning experiments. These datasets are easy to load and are designed to help users quickly prototype, train, and evaluate their deep learning models.*

#### Business Goal
*The business goal is to build and deploy an effective machine learning model using the dataset from the Keras API. This model should deliver accurate predictions or classifications, helping to solve a specific business problem, improve decision-making, enhance user experience, drive operational efficiency, or create new revenue opportunities.*

#### Goal Approach
**1. Dataset**:
The IMDB dataset contains 50,000 movie reviews, divided into training and test sets. Each review is labeled as either "positive" (1) or "negative" (0).

**2. Preprocessing**:
Before feeding the reviews into the model:

`Tokenization`: Text is split into individual words or tokens.

`Padding`: Since reviews can have different lengths, padding ensures that all input sequences have the same length (max_len).

`Vocabulary Indexing`: Words are mapped to integer indices using a tokenizer, where max_words defines the size of the vocabulary.

**3. Model Architecture**:
The model used for sentiment analysis often employs recurrent neural networks (RNNs), like the one you posted:

`Embedding Layer`: Converts the word indices into dense vectors, which can capture semantic information about words.

`Bidirectional GRU Layer`: The GRU (Gated Recurrent Unit) processes the sequences in both directions (forward and backward) to better understand the context.

`Dropout Layers`: These prevent the model from overfitting by randomly dropping a percentage of units during training.

`Dense Output Layer`: The sigmoid activation function produces a probability, indicating whether the review is positive (1) or negative (0).

**4. Training**:
The model is trained using the training dataset, and the labels (positive or negative) are used to calculate the loss and adjust the weights using backpropagation.

**5. Evaluation**:
Once trained, the model is evaluated on the test set. Accuracy, precision, recall, or F1-score can be used to assess its performance.

**6. Use Case**:
This type of analysis is useful for automatically categorizing user reviews, understanding customer sentiment, and enabling businesses to respond to customer feedback efficiently.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Bidirectional, GRU, Dropout
import warnings
warnings.filterwarnings('ignore')

In [2]:
!wget -O dataset.csv 'https://github.com/Subrat1920/IMDB-Sentiment-Analysis/raw/refs/heads/main/Notebook/Datasets/cleaned_sentiment_data.csv'

--2025-04-29 03:39:38--  https://github.com/Subrat1920/IMDB-Sentiment-Analysis/raw/refs/heads/main/Notebook/Datasets/cleaned_sentiment_data.csv
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/Subrat1920/IMDB-Sentiment-Analysis/refs/heads/main/Notebook/Datasets/cleaned_sentiment_data.csv [following]
--2025-04-29 03:39:39--  https://raw.githubusercontent.com/Subrat1920/IMDB-Sentiment-Analysis/refs/heads/main/Notebook/Datasets/cleaned_sentiment_data.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 57950796 (55M) [text/plain]
Saving to: ‘dataset.csv’

dataset.csv         100%[================

In [3]:
df = pd.read_csv('dataset.csv')
df.drop(columns=['Unnamed: 0'], inplace=True)
df.head()

,Reviews,Sentiments
0,this film was just brilliant casting location ...,1
1,big hair big boobs bad music and a giant safet...,0
2,this has to be one of the worst films of the 1...,0
3,the at storytelling the traditional sort man...,1
4,worst mistake of my life i picked this movie...,0


In [4]:
print(f'Shape of the dataset is : {df.shape}')

Shape of the dataset is : (50000, 2)


In [5]:
print('Sample review')
print('-'*50)
print(df.iloc[0,0])
print('-'*50)
print(df.iloc[0,1])

Sample review
--------------------------------------------------
this film was just brilliant casting location scenery story direction everyones really suited the part they played and you could just imagine being there robert  is an amazing actor and now the same being director  father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for  and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also  to the two little boys that played the  of norman and paul they were just brilliant children are often left out of the  list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and 

In [6]:
## Natural Language Processing
import nltk, re, string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [7]:
nltk.download('stopwords')
nltk.download('puckt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Error loading puckt: Package 'puckt' not found in index
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [8]:
stop_words = set(stopwords.words('english'))
len(stop_words)

198

In [9]:
## preprocess the data
def clean_text(text):
  text = text.lower()
  text = re.sub(f"[{string.punctuation}]", "", text)
  text = re.sub(r'\d+', '', text)
  text = ' '.join([word for word in text.split() if word not in stop_words])
  return text

In [10]:
df['processed_revs'] = df["Reviews"].apply(clean_text)

In [11]:
df.head()

,Reviews,Sentiments,processed_revs
0,this film was just brilliant casting location ...,1,film brilliant casting location scenery story ...
1,big hair big boobs bad music and a giant safet...,0,big hair big boobs bad music giant safety pin ...
2,this has to be one of the worst films of the 1...,0,one worst films friends watching film target a...
3,the at storytelling the traditional sort man...,1,storytelling traditional sort many years event...
4,worst mistake of my life i picked this movie...,0,worst mistake life picked movie target figured...


In [12]:
sentences = df["processed_revs"].tolist()
max_len = max(len(x.split()) for x in sentences)
max_len

1137

In [13]:
total_sentences = ', '.join(sentences)
words_present = total_sentences.split(' ')

In [14]:
max_words = len(words_present)
print(f'We have a logged {len(words_present)} number of words in the whole dataset.')

We have a logged 5339017 number of words in the whole dataset.


In [15]:
vocabulary = set(words_present)
vocabulary_size = len(vocabulary)
print(f'So we are dealing with {vocabulary_size} vocabulary size.')

So we are dealing with 15517 vocabulary size.


In [16]:
print('Sample of a vocabulary')
list(vocabulary)[:5]

Sample of a vocabulary


['assistance', 'toward', 'centers', 'fairbanks', 'critique']

In [17]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [18]:
tokenizer = Tokenizer(num_words=vocabulary_size+1, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)

In [19]:
print(f'Number of word with index logged : {len(tokenizer.word_counts)}')

Number of word with index logged : 9610


In [20]:
sequences = tokenizer.texts_to_sequences(sentences)

In [21]:
## check if there is equal words present in it
len(sequences[0]) == len(sentences[0].split())

True

In [22]:
## padding sequences
padded_sequence = pad_sequences(sequences, maxlen=max_len, padding='post')

In [23]:
## padded sequence for random line and total paded sequences present
len(padded_sequence[10]), len(padded_sequence)

(1137, 50000)

In [24]:
from tensorflow.keras.layers import Embedding, Input
from tensorflow.keras.models import Model

In [25]:
## Applying Embeddings
embedding_dim = 10
input_layer = Input(shape=(max_len,))
embedding_layer = Embedding(input_dim=vocabulary_size, output_dim=embedding_dim, input_length=max_len)(input_layer)
embedding_model = Model(inputs=input_layer, outputs=embedding_layer)

In [26]:
embedding_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1137)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 1137, 10)       │       155,170 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 155,170 (606.13 KB)

 Trainable params: 155,170 (606.13 KB)

 Non-trainable params: 0 (0.00 B)

In [27]:
padded_sequence.shape

(50000, 1137)

In [28]:
padded_sequence[0]

array([  3, 398, 919, ...,   0,   0,   0], dtype=int32)

In [29]:
## let's try to convert the first padded sequence into embedings
embedding_model.predict(np.array([padded_sequence[0]]))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 312ms/step


array([[[ 0.0361364 , -0.02924824,  0.01106571, ...,  0.02482151,
         -0.01247789,  0.00587802],
        [ 0.02586483, -0.01904371,  0.04512474, ..., -0.00806425,
         -0.01032783,  0.02636118],
        [ 0.01346442, -0.03212177, -0.03362333, ..., -0.04709277,
         -0.0351317 , -0.01510924],
        ...,
        [-0.03115041, -0.01955576,  0.01819123, ...,  0.01559296,
          0.02607734, -0.00958972],
        [-0.03115041, -0.01955576,  0.01819123, ...,  0.01559296,
          0.02607734, -0.00958972],
        [-0.03115041, -0.01955576,  0.01819123, ...,  0.01559296,
          0.02607734, -0.00958972]]], dtype=float32)

In [30]:
embeddings = embedding_model.predict(padded_sequence)

1563/1563 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step


In [31]:
embeddings.shape

(50000, 1137, 10)

In [32]:
type(embeddings)

numpy.ndarray

In [33]:
np.savez_compressed('embeddings_compressed.npz', embeddings=embeddings)

In [34]:
from google.colab import files
files.download("embeddings_compressed.npz")
print('Embeddings layer downloaded')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Embeddings layer downloaded


In [35]:
embedding_model.save("embedding_model.keras")
files.download("embedding_model.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>